In [1]:
import fastf1
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

CACHE_DIR = Path.home() / "Library/Caches/fastf1"
DATA_DIR  = Path("~/Downloads/F1 race intelligence/data").expanduser()
DATA_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

TODAY = datetime.today()
print(f"FastF1 version: {fastf1.__version__}")
print(f"Data will be saved to: {DATA_DIR}")

FastF1 version: 3.8.3
Data will be saved to: /Users/sanjaychowdary/Downloads/F1 race intelligence/data


In [2]:
def get_reg_era(year: int) -> int:
    """
    0 = Pre-ground effect (≤2021)
    1 = Ground effect era (2022–2025)
    2 = 2026 new regs
    # When 2027 arrives → add: elif year <= 2027: return 3
    """
    if year <= 2021:   return 0
    elif year <= 2025: return 1
    else:              return 2

In [3]:
import time

def collect_race(year: int, round_num: int, event_name: str, session_type: str = 'R'):
    try:
        session = fastf1.get_session(year, round_num, session_type)
        session.load(laps=True, telemetry=False, weather=True, messages=True)
        laps = session.laps.copy()

        if laps.empty:
            return None

        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
        laps['Season']         = year
        laps['Round']          = round_num
        laps['EventName']      = event_name
        laps['SessionType']    = session_type
        laps['RegEra']         = get_reg_era(year)
        laps['Circuit']        = session.event['Location']

        location_map = {
            'Miami Gardens' : 'Miami',
            'Yas Island'    : 'Yas Marina',
            'Marina Bay'    : 'Singapore',
            'Monte Carlo'   : 'Monaco',
        }
        laps['Circuit'] = laps['Circuit'].replace(location_map)

        # Lap time filter — relaxed for practice (out laps, slow runs)
        min_time = 60 if session_type in ['R', 'S', 'SS', 'Q'] else 55
        laps = laps[
            (laps['LapTimeSeconds'] > min_time) &
            (laps['LapTimeSeconds'] < 200) &
            (laps['IsAccurate'] == True)
        ].copy()

        if laps.empty:
            return None

        laps['LapTimeZ'] = laps.groupby('Driver')['LapTimeSeconds'].transform(
            lambda x: (x - x.mean()) / max(x.std(), 0.001)
        )

        laps['UnderSC']  = False
        laps['UnderVSC'] = False
        try:
            track_status = session.track_status
            if track_status is not None and len(track_status) > 0:
                for _, row in track_status.iterrows():
                    status   = str(row.get('Status', ''))
                    lap_time = row.get('Time', None)
                    if lap_time is not None:
                        matching = laps[laps['LapStartTime'] <= lap_time]
                        if len(matching) > 0:
                            lap_num = int(matching['LapNumber'].max())
                            if status == '4':
                                laps.loc[laps['LapNumber'] == lap_num, 'UnderSC']  = True
                            elif status == '6':
                                laps.loc[laps['LapNumber'] == lap_num, 'UnderVSC'] = True
        except:
            pass

        cols = [
            'Season', 'Round', 'EventName', 'Circuit', 'RegEra',
            'SessionType', 'Driver', 'Team', 'LapNumber', 'LapTimeSeconds',
            'LapTimeZ', 'Compound', 'TyreLife', 'FreshTyre', 'UnderSC', 'UnderVSC'
        ]
        cols = [c for c in cols if c in laps.columns]
        return laps[cols]

    except Exception as e:
        raise e

In [4]:
import time
from datetime import datetime
from fastf1.exceptions import RateLimitExceededError

all_data   = []
CURRENT_YEAR = datetime.today().year
SEASONS    = range(2018, CURRENT_YEAR + 1)  # ← dynamic, works for 2027, 2028, forever
SLEEP_SEC  = 3
SESSION_TYPES = ['R', 'S', 'Q', 'FP1', 'FP2', 'FP3']

for year in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
    except RateLimitExceededError:
        print(f"⏳ Rate limited — waiting 60s...")
        time.sleep(60)
        try:
            schedule = fastf1.get_event_schedule(year, include_testing=False)
        except Exception as e:
            print(f"❌ {year}: {e}")
            continue
    except Exception as e:
        print(f"⚠️  {year}: {e}")
        continue

    print(f"\n{'='*55}")
    print(f"  📅 Season {year} — {len(schedule)} events")
    print(f"{'='*55}")

    for _, event in schedule.iterrows():
        event_date = pd.to_datetime(event['EventDate'])
        event_name = event['EventName']
        round_num  = int(event.get('RoundNumber', _))

        if event_date > pd.Timestamp(TODAY):
            print(f"  ⏭️  Skipping future: {event_name}")
            continue

        for session_type in SESSION_TYPES:
            print(f"  🏎️  {event_name} {year} [{session_type}]...", end='', flush=True)
            retries = 3
            for attempt in range(retries):
                try:
                    df = collect_race(year, round_num, event_name, session_type)
                    if df is not None:
                        all_data.append(df)
                        print(f" ✅ {len(df)} laps")
                    else:
                        print(f" ⚠️  no valid laps")
                    break
                except RateLimitExceededError:
                    wait = 60 * (attempt + 1)
                    print(f" ⏳ Rate limited (attempt {attempt+1}/{retries}) — waiting {wait}s...")
                    time.sleep(wait)
                except Exception as e:
                    print(f" ❌ {e}")
                    break
            time.sleep(SLEEP_SEC)

print(f"\n✅ Collection complete — {len(all_data)} sessions collected")


  📅 Season 2018 — 21 events
  🏎️  Australian Grand Prix 2018 [R]...

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']


 ✅ 736 laps
  🏎️  Australian Grand Prix 2018 [S]... ❌ Session type 'S' does not exist for this event
  🏎️  Australian Grand Prix 2018 [Q]...

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '7', '5', '33', '3', '20', '8', '27', '55', '77', '14', '2', '11', '18', '31', '28', '9', '16', '35', '10']


 ✅ 14 laps
  🏎️  Australian Grand Prix 2018 [FP1]...

...

 ⏳ Rate limited (attempt 2/3) — waiting 120s...


core           INFO 	Loading data for Miami Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 22 drivers: ['12', '3', '16', '1', '63', '44', '81', '43', '10', '27', '30', '87', '55', '31', '23', '

 ✅ 116 laps
  🏎️  Miami Grand Prix 2026 [FP1]...

core           INFO 	Loading data for Miami Grand Prix - Practice 1 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	D

 ✅ 499 laps
  🏎️  Miami Grand Prix 2026 [FP2]... ❌ Session type 'FP2' does not exist for this event
  🏎️  Miami Grand Prix 2026 [FP3]... ❌ Session type 'FP3' does not exist for this event
  ⏭️  Skipping future: Canadian Grand Prix
  ⏭️  Skipping future: Monaco Grand Prix
  ⏭️  Skipping future: Barcelona Grand Prix
  ⏭️  Skipping future: Austrian Grand Prix
  ⏭️  Skipping future: British Grand Prix
  ⏭️  Skipping future: Belgian Grand Prix
  ⏭️  Skipping future: Hungarian Grand Prix
  ⏭️  Skipping future: Dutch Grand Prix
  ⏭️  Skipping future: Italian Grand Prix
  ⏭️  Skipping future: Spanish Grand Prix
  ⏭️  Skipping future: Azerbaijan Grand Prix
  ⏭️  Skipping future: Singapore Grand Prix
  ⏭️  Skipping future: United States Grand Prix
  ⏭️  Skipping future: Mexico City Grand Prix
  ⏭️  Skipping future: São Paulo Grand Prix
  ⏭️  Skipping future: Las Vegas Grand Prix
  ⏭️  Skipping future: Qatar Grand Prix
  ⏭️  Skipping future: Abu Dhabi Grand Prix

✅ Collection complete — 829 sessi

In [5]:
# ── Save to parquet ───────────────────────────────────────────────────────────
final = pd.concat(all_data, ignore_index=True)
out   = DATA_DIR / "all_races_2018_2026.parquet"
final.to_parquet(out, index=False)

print(f"✅ Saved → {out}")
print(f"\n📊 Dataset Summary:")
print(f"   Total laps  : {len(final):,}")
print(f"   Seasons     : {sorted(final['Season'].unique())}")
print(f"   Races       : {final['EventName'].nunique()}")
print(f"   Drivers     : {final['Driver'].nunique()} — {sorted(final['Driver'].unique())}")
print(f"   Circuits    : {final['Circuit'].nunique()}")
print(f"   Compounds   : {final['Compound'].unique()}")
print(f"   Reg Eras    : {final['RegEra'].value_counts().to_dict()}")
print(f"\n   File size   : {out.stat().st_size / 1024**2:.1f} MB")

✅ Saved → /Users/sanjaychowdary/Downloads/F1 race intelligence/data/all_races_2018_2026.parquet

📊 Dataset Summary:
   Total laps  : 325,983
   Seasons     : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   Races       : 36
   Drivers     : 69 — ['AIT', 'ALB', 'ALO', 'ANT', 'ARO', 'BEA', 'BEG', 'BOR', 'BOT', 'BRO', 'COL', 'CRA', 'DEN', 'DEV', 'DOO', 'DRU', 'DUN', 'ERI', 'FIT', 'FUO', 'GAS', 'GEL', 'GIO', 'GRO', 'HAD', 'HAM', 'HAR', 'HIR', 'HUL', 'ILO', 'IWA', 'KUB', 'KVY', 'LAT', 'LAW', 'LEC', 'LEL', 'LIN', 'MAG', 'MAR', 'MAZ', 'MSC', 'NIS', 'NOR', 'OCO', 'OSU', 'OWA', 'PAL', 'PER', 'PIA', 'POU', 'RAI', 'RIC', 'RUS', 'SAI', 'SAR', 'SCH', 'SHI', 'SHW', 'SIR', 'STR', 'TSU', 'VAN', 'VER', 'VES', 'VET', 'VIP', 'YAM', 'ZHO']
   Circuits    : 31
   Compounds   : ['ULTRASOFT' 'SOFT' 'SUPERSOFT' 'INTERMEDIATE' 'MEDIUM' 'None' 'nan'
 'HYPERSOFT' 'WET' 'HARD' 'TEST' 'TEST_UNKNOWN' 'UNKNOWN' '']
   